# Customer Churn: Responsible Baseline Notebook

Educational exercise for the internship task. This CSV has only 12 rows; all scores are unstable and must not be treated as real-world evidence. Place `customer-churn-training.csv` in the same folder as this notebook before running.


In [3]:
import pandas as pd
from IPython.display import display

df = pd.read_csv('customer-churn-training.csv')
print('Shape:', df.shape)
display(df.head())
display(df.dtypes)


Shape: (12, 7)


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


,0
customer_id,object
tenure_months,int64
support_tickets,int64
monthly_spend_inr,int64
last_login_days,int64
plan_type,object
churned,int64


## 1. Quality checks


In [4]:
print('Missing values by column:')
display(df.isna().sum())
print('Exact duplicate rows:', df.duplicated().sum())
print('Target counts:')
display(df['churned'].value_counts().sort_index())
print('Plan type vs churn:')
display(pd.crosstab(df['plan_type'], df['churned']))
display(df.describe(include='all').T)


Missing values by column:


,0
customer_id,0
tenure_months,0
support_tickets,0
monthly_spend_inr,0
last_login_days,0
plan_type,0
churned,0


Exact duplicate rows: 0
Target counts:


,count
churned,
0,7
1,5


Plan type vs churn:


churned,0,1
plan_type,,
Basic,0,5
Pro,3,0
Standard,4,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,12,12,C001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure_months,12.0,NaN,NaN,NaN,12.166667,9.242524,1.0,5.5,9.5,18.5,30.0
support_tickets,12.0,NaN,NaN,NaN,2.25,1.602555,0.0,1.0,2.0,3.25,5.0
monthly_spend_inr,12.0,NaN,NaN,NaN,915.666667,332.574895,499.0,724.0,899.0,1074.0,1499.0
last_login_days,12.0,NaN,NaN,NaN,10.0,9.095453,1.0,2.75,7.0,14.75,30.0
plan_type,12,3,Basic,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
churned,12.0,NaN,NaN,NaN,0.416667,0.514929,0.0,0.0,0.0,1.0,1.0


## 2. Prepare features and split data
`customer_id` is an identifier, so it is excluded. Preprocessing is placed inside the pipeline to avoid fitting it on the test data. The split is for demonstration only because there are so few rows.


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score

target = 'churned'
X = df.drop(columns=['customer_id', target])
y = df[target]
numeric_features = ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']
categorical_features = ['plan_type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)
print('Train rows:', len(X_train), '| Test rows:', len(X_test))


Train rows: 8 | Test rows: 4


## 3. Majority-class baseline


In [6]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print('Baseline confusion matrix:\n', confusion_matrix(y_test, baseline_pred))
print(classification_report(y_test, baseline_pred, zero_division=0))
print('Balanced accuracy:', round(balanced_accuracy_score(y_test, baseline_pred), 3))


Baseline confusion matrix:
 [[2 0]
 [2 0]]
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         2
           1       0.00      0.00      0.00         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4

Balanced accuracy: 0.5


## 4. Simple Logistic Regression model


In [7]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocess = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])
model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Model confusion matrix:\n', confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))
print('Balanced accuracy:', round(balanced_accuracy_score(y_test, pred), 3))


Model confusion matrix:
 [[2 0]
 [0 2]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4

Balanced accuracy: 1.0


## 5. Interpretation and limitations
- Compare the model to the majority-class baseline; a more complex model is not automatically better.
- Read the confusion matrix and note false positives and false negatives.
- In this file, plan type perfectly separates churn labels. Investigate this pattern; do not assume it will hold in real data.
- Only 12 rows are present. A different split could change the scores substantially.
- Provenance, permission, label timing, and feature timing are not documented in the supplied files. Confirm them before any real-world use.
- Do not use predictions to deny service, change prices, or penalize customers. Keep a human in the loop.
